# Trening osobno: eksperymenty poza pipeline'em

Ten notebook to trening z pipeline'u wyjęty na wierzch, w osobnym katalogu `trening/`, żeby
eksperymentować bez dotykania `rung9_hpo.ipynb` ani obrazu. Dane, podział na foldy, epizody testowe,
podłoga i próg bramki są **te same** co w kampanii (ten sam loader, ten sam `split.json`, ten sam seed),
więc liczba stąd znaczy dokładnie to samo, co liczba z tabeli Experiments. Różnica: model, epizod i pętla
treningowa są tu zwykłymi komórkami, które zmieniasz.

Komórki:

- **KONFIGURACJA** — fold, split, urządzenie (CPU, jak kontener w kampanii — GPU daje inne zaokrąglenia i liczby
  nieporównywalne z tabelą Experiments), czy logować do Experiments, notatka do eksperymentu.
- **DANE** (nie zmieniaj) — ładuje freeze i `split.json`, buduje samplery, czyta podłogę i próg.
- **MODEL** (zmieniaj) — enkoder; kopia `fsl/models/mlp.py`.
- **EPIZOD** (zmieniaj) — jak liczy się prototyp, odległość i strata; kopia `fast_adapt`.
- **TRENING** (zmieniaj) — optymalizator, lr, liczba iteracji, harmonogram.
- **OCENA** (nie zmieniaj) — 1000 epizodów testowych, podłoga, próg, lift, werdykt bramki; wiersz w tabeli
  `WYNIKI` i w `wyniki/wyniki_f<fold>.csv` (katalog `wyniki/` nie wchodzi do gita).
- **SAMOKONTROLA** — uruchom raz przed pierwszą zmianą: pętla z tego notebooka musi dać bit w bit to samo,
  co `fsl.training.loop.train()` z pipeline'u. Jeśli nie daje, porównania poniżej nic nie znaczą.

Reguły, żeby nie oszukać samej siebie:

1. Kryterium przed uruchomieniem: co ma się poprawić, o ile, na czym. Wpisz je w `NOTATKA`.
2. Wybieraj po **walidacji** (`val_last`, typologie walidacyjne foldu). Test liczy się i tak, ale decyzję
   podejmuj po val; test potwierdza raz, na końcu.
3. Jeden fold to jedna próbka. Kandydat, który wygrał na foldzie 3, idzie na foldy 0–4 (zmień `FOLD`,
   uruchom od DANE). Ostatecznym sędzią jest kampania: zwycięzca staje się enkoderem w
   `fsl/models/registry.py` i przechodzi przez bramkę z resztą pipeline'u.
4. Jeśli zmieniasz odległość lub stratę w EPIZOD, pamiętaj, że bramka w pipelinie mierzy `fast_adapt`
   (euklides). Taki kandydat przed promocją potrzebuje tej samej zmiany w `fsl/models/protonet.py`.


In [ ]:
# --- KONFIGURACJA --------------------------------------------------------------
PROJECT, REGION, BUCKET = "dark-data-discovery", "us-central1", "dark-data-discovery-fsl-data"
DATASET = "samld-case"
FOLD = 3                       # 0..4; fold 3 = fold demo
SPLIT_URI = ""                 # split.json z kampanii (gs://.../splits/samld-case/<sha16>/split.json);
                               # puste = najnowszy split w buckecie; brak splitu = foldy i podlogi z freeze'u
SEED = 0
DEVICE = "cpu"                 # jak kontener treningowy w kampanii. GPU liczy w innej kolejnosci (inne zaokraglenia):
                               # po 600 krokach inne wagi, wynik nieporownywalny z tabela Experiments i nie zawsze powtarzalny
EXPERIMENT_NAME = "fsl-samld-case"
LOG_TO_EXPERIMENTS = False     # True: kazdy eksperyment = run "notebook-f<fold>-<czas>" w tej samej tabeli co kampanie
NOTATKA = "baseline: MLP 2x64, euklides, Adam 1e-3, 600 iteracji"   # kryterium i opis; trafia do tabeli i do runu


In [ ]:
# --- DANE (nie zmieniaj) ---------------------------------------------------------
import json, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from fsl.config import TrainConfig
from fsl.data.manifest import fetch_manifest, get_geometry
from fsl.data.samld import build_task_samplers, load_frozen_samld
from fsl.training.loop import evaluate, set_seed

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/pipeline_config_samld.json").exists())
CFG = json.loads((ROOT / "configs/pipeline_config_samld.json").read_text())
BASE = CFG["training"]                                  # te same wartosci, ktore dostaje kontener treningowy
MAX_STD = CFG["evaluation"]["max_std_threshold"]
WYNIKI_DIR = Path("wyniki")                             # obok notebooka; w .gitignore
WYNIKI_DIR.mkdir(exist_ok=True)

# projekt eksperymentu z pipeline'u (v1.1+): split.json zapisany przez prepare_experiment
ma_split = "split_uri" in TrainConfig.__dataclass_fields__
if ma_split and not SPLIT_URI:                          # najnowszy split.json w buckecie
    from google.cloud import storage
    docs = [b for b in storage.Client(project=PROJECT).list_blobs(BUCKET, prefix=f"splits/{DATASET}/")
            if b.name.endswith("/split.json")]
    if docs:
        SPLIT_URI = f"gs://{BUCKET}/{max(docs, key=lambda b: b.updated).name}"
if SPLIT_URI and not ma_split:
    raise SystemExit("ta wersja pakietu fsl nie zna split_uri — zaktualizuj repo (v1.1+) albo ustaw SPLIT_URI = ''")

manifest = fetch_manifest(PROJECT, BUCKET, DATASET)
in_features, _ = get_geometry(manifest, DATASET)
cfg = TrainConfig(project=PROJECT, region=REGION, bucket=BUCKET, dataset=DATASET, fold=FOLD,
                  encoder="mlp", in_channels=in_features, image_size=1, seed=SEED,
                  n_way=BASE["n_way"], k_shot=BASE["k_shot"], query=BASE["query"],
                  train_iters=BASE["train_iters"], eval_every=BASE["eval_every"],
                  val_episodes=BASE["val_episodes"], test_episodes=BASE["test_episodes"],
                  lr=BASE["lr"], embedding_hid=BASE["embedding_hid"],
                  mlp_depth=BASE["mlp_depth"], mlp_dropout=BASE["mlp_dropout"],
                  experiment_name=EXPERIMENT_NAME, log_to_vertex=False,
                  **({"split_uri": SPLIT_URI} if ma_split else {}))
bundle, data_sha = load_frozen_samld(cfg)
spec = bundle["folds"][FOLD]

if SPLIT_URI:                                           # podloga i prog tego foldu, te same co w bramce kampanii
    from fsl.data.split import load_split
    split_doc = load_split(SPLIT_URI, PROJECT)
    FLOOR, THRESHOLD, SPLIT_SHA = split_doc["floors"][str(FOLD)], split_doc["thresholds"][str(FOLD)], split_doc["split_sha"]
else:
    legacy = Path.home() / "data_cache/samld_case_floors.json"
    assert legacy.exists(), "brak split.json w buckecie i brak ~/data_cache/samld_case_floors.json — uruchom kampanie albo komorke 3 samld_case_freeze.ipynb"
    fl = json.loads(legacy.read_text())
    FLOOR, THRESHOLD, SPLIT_SHA = fl["floors"][str(FOLD)], fl["thresholds"][str(FOLD)], ""

device = torch.device(DEVICE)
WYNIKI = []                                             # jeden wiersz na eksperyment w tej sesji


def samplery():
    # swieze samplery = te same epizody za kazdym razem (seed); wolaj przed kazdym treningiem i ocena
    return build_task_samplers(bundle, cfg)


print(f"{DATASET} | fold {FOLD} | split {SPLIT_SHA[:12] or 'freeze'} | dane {data_sha[:12]} | {in_features} cech | {device}")
print(f"test (nowe dla modelu): {', '.join(spec['test'])}")
print(f"val (do wyboru kandydata): {', '.join(spec['val'])}")
print(f"train: {len(spec['train'])} typologii")
print(f"podloga {FLOOR:.4f} | prog bramki {THRESHOLD:.4f} (dolne CI) | max std {MAX_STD}")


In [ ]:
# --- MODEL (zmieniaj) ------------------------------------------------------------
import torch.nn as nn


class Encoder(nn.Module):
    """Kopia fsl/models/mlp.py. Wejscie (B, F) -> osadzenie (B, hid).
    Zmieniaj: glebokosc, szerokosc, normalizacje, dropout, aktywacje, polaczenia skrotowe."""

    def __init__(self, in_features: int, hid: int = 64, depth: int = 2, dropout: float = 0.0):
        super().__init__()
        layers, width = [], in_features
        for _ in range(depth):
            layers += [nn.Linear(width, hid), nn.LayerNorm(hid), nn.ReLU()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            width = hid
        layers.append(nn.Linear(width, hid))            # glowica osadzenia, bez aktywacji
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x.flatten(1))


def zbuduj_model():
    return Encoder(in_features, hid=BASE["embedding_hid"], depth=BASE["mlp_depth"], dropout=BASE["mlp_dropout"])


In [ ]:
# --- EPIZOD (zmieniaj) -----------------------------------------------------------
import torch.nn.functional as F


def episode(model, batch, ways, shot, query, device):
    """Kopia fast_adapt z fsl/models/protonet.py: jeden epizod -> (strata, trafnosc).
    Prototyp = srednia osadzen wsparcia; logity = -odleglosc euklidesowa; strata = entropia krzyzowa.
    Zmieniaj: odleglosc (np. kosinus z temperatura), strate, sposob liczenia prototypu."""
    data, labels = batch
    data, labels = data.to(device), labels.to(device)
    order = torch.sort(labels, stable=True)
    data, labels = data[order.indices], labels[order.indices]

    emb = model(data)
    support = np.zeros(data.size(0), dtype=bool)
    first = np.arange(ways) * (shot + query)
    for offset in range(shot):
        support[first + offset] = True
    support_t = torch.from_numpy(support).to(device)
    query_t = torch.from_numpy(~support).to(device)

    prototypes = emb[support_t].reshape(ways, shot, -1).mean(dim=1)
    q, q_labels = emb[query_t], labels[query_t].long()

    logits = -torch.cdist(q, prototypes)                # <- np. kosinus: TEMP * F.normalize(q) @ F.normalize(prototypes).T
    loss = F.cross_entropy(logits, q_labels)
    acc = (logits.argmax(1) == q_labels).float().mean()
    return loss, acc


def ocena(model, tasks, n_episodes):
    """Trafnosc na n epizodach z samplera, TYM SAMYM epizodem co w treningu (to, co trenujesz, to mierzysz)."""
    model.eval()
    accs = []
    with torch.no_grad():
        for _ in range(n_episodes):
            _, acc = episode(model, tasks.sample(), cfg.n_way, cfg.k_shot, cfg.query, device)
            accs.append(acc.item())
    model.train()
    return np.array(accs)


In [ ]:
# --- TRENING (zmieniaj) ----------------------------------------------------------
ITERS, LR, EVAL_EVERY = BASE["train_iters"], BASE["lr"], BASE["eval_every"]


def trenuj(verbose=True):
    """Ta sama kolejnosc co fsl.training.loop.train(): seed -> model -> optymalizator -> seed -> petla.
    Zmieniaj: optymalizator, lr i harmonogram, liczbe iteracji, regularyzacje, wczesne zatrzymanie po val."""
    train_tasks, val_tasks, _ = samplery()
    set_seed(SEED)
    model = zbuduj_model().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    set_seed(SEED)
    krzywa, run_loss, run_acc, seen = [], 0.0, 0.0, 0
    t0 = time.time()
    for it in range(1, ITERS + 1):
        opt.zero_grad()
        loss, acc = episode(model, train_tasks.sample(), cfg.n_way, cfg.k_shot, cfg.query, device)
        loss.backward()
        opt.step()
        run_loss, run_acc, seen = run_loss + loss.item(), run_acc + acc.item(), seen + 1
        if it % EVAL_EVERY == 0 or it == 1:
            val = ocena(model, val_tasks, BASE["val_episodes"])
            krzywa.append({"iter": it, "train_loss": run_loss / seen, "train_acc": run_acc / seen,
                           "val_acc": float(val.mean())})
            if verbose:
                print(f"iter {it:5d} | train loss {run_loss / seen:.3f} acc {run_acc / seen:.3f} | val acc {val.mean():.3f}")
            run_loss, run_acc, seen = 0.0, 0.0, 0
    return model, krzywa, time.time() - t0


In [ ]:
# --- OCENA (nie zmieniaj) --------------------------------------------------------
model, krzywa, czas = trenuj()
_, _, test_tasks = samplery()
test = ocena(model, test_tasks, BASE["test_episodes"])
mean, std = float(test.mean()), float(test.std())
ci = float(1.96 * std / np.sqrt(len(test)))
lower, lift = mean - ci, mean - FLOOR
passed = lower >= THRESHOLD and std <= MAX_STD
val_last = krzywa[-1]["val_acc"]
val_best = max(k["val_acc"] for k in krzywa)
print(f"\nTEST fold {FOLD}: {mean:.4f} +/- {ci:.4f} (std {std:.3f}) | dolne CI {lower:.4f} vs prog {THRESHOLD:.4f} "
      f"| podloga {FLOOR:.4f} | lift {lift:+.4f} | bramka {'PASS' if passed else 'FAIL'}")
print(f"val: ostatnia {val_last:.4f}, najlepsza {val_best:.4f} | {czas:.0f} s")

wiersz = {"czas": time.strftime("%Y-%m-%d %H:%M"), "notatka": NOTATKA, "fold": FOLD, "split": SPLIT_SHA[:12] or "freeze",
          "iters": ITERS, "lr": LR, "val_last": round(val_last, 4), "val_best": round(val_best, 4),
          "test": round(mean, 4), "ci95": round(ci, 4), "lift": round(lift, 4), "bramka": "PASS" if passed else "FAIL"}
WYNIKI.append(wiersz)
plik = WYNIKI_DIR / f"wyniki_f{FOLD}.csv"               # dopisuje wiersz; historia miedzy sesjami
pd.DataFrame([wiersz]).to_csv(plik, mode="a", header=not plik.exists(), index=False)

if LOG_TO_EXPERIMENTS:                                  # ten sam zapis co kontener treningowy (fsl.training.loop)
    from fsl.training.loop import ExperimentLog
    log = ExperimentLog(cfg, run_name=f"notebook-f{FOLD}-{int(time.time())}", enabled=True,
                        extra_params={"run_type": "notebook", "note": NOTATKA, "split_sha": SPLIT_SHA,
                                      "split_uri": SPLIT_URI, "train_iters": ITERS, "lr": LR})
    log.start(data_sha)
    for k in krzywa:
        log.curve({"train_loss": k["train_loss"], "train_acc": k["train_acc"], "val_acc": k["val_acc"]}, step=k["iter"])
    log.finish({"test_accuracy_mean": mean, "test_accuracy_ci95": ci, "test_accuracy_std": std,
                "train_time_seconds": czas, "floor": FLOOR, "gate_threshold": THRESHOLD, "lift": lift,
                "accuracy_lower_ci": lower, "gate_passed": 1.0 if passed else 0.0, "val_acc_last": val_last})
    print("run:", log.run_name or "(nie zalogowano, patrz WARNING wyzej)")

pd.DataFrame(WYNIKI)


In [ ]:
# --- SAMOKONTROLA (uruchom raz, przy niezmienionych komorkach MODEL/EPIZOD/TRENING) ---------
# Pipeline trenuje przez fsl.training.loop.train(); ta sama konfiguracja musi dac identyczny wynik.
from fsl.training.loop import train as train_pipeline

ref = train_pipeline(cfg)                                # log_to_vertex=False: nic nie zapisuje
_, _, t_nb = samplery()
_, _, t_ref = samplery()
nb_acc = float(ocena(model, t_nb, 200).mean())           # model z komorki OCENA, swieze epizody
ref_acc = float(evaluate(ref["model"], t_ref, cfg, 200, device).mean())
same_weights = all(torch.equal(a, b) for a, b in zip(model.state_dict().values(), ref["model"].state_dict().values()))
print(f"wagi identyczne: {same_weights} | 200 epizodow: notebook {nb_acc:.4f} vs pipeline {ref_acc:.4f} | "
      f"test pipeline'u {ref['test_accuracy_mean']:.4f} vs notebook {mean:.4f}")
print("OK: notebook = pipeline" if same_weights and abs(ref["test_accuracy_mean"] - mean) < 1e-9
      else "ROZNICA: ktoras komorka odbiega od pipeline'u — nie porownuj wynikow, dopoki to nie jest OK")


## Od czego zacząć

Trzy zmiany, każda osobno, każda z kryterium wpisanym w `NOTATKA` przed uruchomieniem (np.
„val_last ≥ baseline + 0.03 na foldzie 3, potem ≥ 4/5 foldów”):

1. **Harmonogram i czas** (TRENING): `ITERS` 600 → 1500 z `torch.optim.lr_scheduler.CosineAnnealingLR`,
   albo wczesne zatrzymanie po najlepszym `val_acc`. Najtańsza zmiana; mówi, czy model jest niedouczony,
   czy sufit jest gdzie indziej.
2. **Odległość** (EPIZOD): kosinus z temperaturą zamiast euklidesa — jedna linia `logits`. Mechanizm: euklides
   na osadzeniu LayerNorm karze normę, nie kierunek. Przed promocją wymaga tej samej zmiany w `fast_adapt`.
3. **Pojemność i regularyzacja** (MODEL): `depth=3`, `hid=128`, `dropout=0.1`. Podłoga mówi, że losowa projekcja
   tych cech jest już dobra, więc tu spodziewaj się najmniej.

Każdy wynik to wiersz w `WYNIKI` i w `wyniki/wyniki_f<fold>.csv`; z `LOG_TO_EXPERIMENTS=True` także run w tabeli
obok kampanii. Jeśli żadna z trzech zmian nie rusza `val_last` o ≥ 3 pp, to jest wynik, nie porażka: sufit siedzi
w cechach, nie w treningu, i punkt 3 przechodzi na kalibrację i „żadna z tych typologii”.
